<a href="https://colab.research.google.com/github/alxmzr/Colab/blob/main/BTC_24H_Predict.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#https://trading-data-analysis.pro/the-trend-is-your-friend-for-your-trading-and-for-neural-prophet-lagged-regressors-part-5-9e9291f636c

In [ ]:
!pip install tiingo
!pip install neuralprophet[live] --quiet
!pip install Pandas
!pip install numpy
!pip install ta --upgrade

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.5/829.5 kB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.4/145.4 kB 9.8 MB/s eta 0:00:00


In [ ]:
from ta import add_all_ta_features
from ta.utils import dropna
import requests
from datetime import date, datetime, timedelta
import pandas as pd
import numpy as np
from tiingo import TiingoClient
from neuralprophet import NeuralProphet, set_log_level
set_log_level("CRITICAL")
from neuralprophet import set_random_seed
set_random_seed(0)
import matplotlib.pyplot as plt
pd.plotting.register_matplotlib_converters()
%matplotlib inline
from sklearn.metrics import mean_squared_error
import itertools
from ta import add_all_ta_features
from ta.utils import dropna

In [ ]:
config = {
     'api_key': 'e5f6564c2239e6d9724c20e04808f9f5448f0ddb',
     'session': True
}
client = TiingoClient(config)

ticker = 'btcusd'
frequency = '130min'
now = datetime.now()
end_date = now + timedelta(days=1)
history = client.get_crypto_price_history([ticker],
          endDate = end_date,
          resampleFreq = frequency)

# BTC price data
df_hist = pd.DataFrame.from_dict(history[0]['priceData'])

In [ ]:
# Clean NaN values
df_hist = dropna(df_hist)

# Add all ta features to BTC dataframe
df_hist = add_all_ta_features(df_hist, open="open", high="high", low="low", close="close", volume="volume")

In [ ]:
# set up the input data for Neural Prophet with the lagged regressor.

df = pd.DataFrame(df_hist, columns=['date', 'close', 'volume_cmf'])
df.rename(columns = {'date':'ds', 'close':'y'}, inplace = True)
df["ds"] = df["ds"].astype("datetime64[ns]")

# inspect the input dataframe
df.info()

In [ ]:
df['volume_cmf'].head(30)

In [ ]:
df = df.iloc[19:]

In [ ]:
# instantiate the model with parameters derived from previous testing and tuning
m = NeuralProphet(
    n_forecasts = 11,
    n_lags = 16,
    learning_rate = 0.01,
    n_changepoints = 90,
    changepoints_range = 0.95,
    trend_reg = 1,
)
# add the regressor to the model
m = m.add_lagged_regressor(names='volume_cmf')

In [ ]:
# split the data into test and train. # Fit the model with the lagged regressor
# use progress="plot-all" to visualize the training process

df_train, df_test = m.split_df(df, valid_p=0.15)
metrics_train = m.fit(df_train, validation_df=df_test, freq=str(frequency), progress="plot-all")
metrics_train[-1:]

In [ ]:
#Ready to make 24 hour forecast from the 130 minute data (1440/130)  = 11
bars_ahead_forecast = 11

In [ ]:
# instantiate a new model
m = NeuralProphet(
    n_forecasts = 11,
    n_lags = 16,
    learning_rate = 0.01,
    n_changepoints = 90,
    changepoints_range = 0.95,
    trend_reg = 1,
)
m = m.add_lagged_regressor(names='volume_cmf')

In [ ]:
# fit the model with the actual data df. Make the forecast.
metrics = m.fit(df, freq="130min")
#metrics = m.fit(df, freq=str(frequency))
future = m.make_future_dataframe(df, periods=int(bars_ahead_forecast), n_historic_predictions=len(df))
forecast = m.predict(future)

# determine how many data points to use in the plot
predict = forecast.tail(50)

In [ ]:
# Set up the chart data. Add 2 standard deviation lines for possible use in the plot
std = predict['yhat1'].std()
upper2 = predict['yhat1'] + (2 * std)
lower2 = predict['yhat1'] - (2 * std)
actual = predict['y']
fcst = predict['yhat1']
trend = predict['trend']
dates = predict['ds']

In [ ]:
x = dates
y1 = upper2
y2 = actual
y3 = fcst
y4 = lower2
y5 = trend
fig = plt.figure()
fig.set_figwidth(13)
fig.set_figheight(8)

plt.title('BITCOIN 24 HOUR FORECAST', fontsize = 25)
plt.xlabel('Month-Day-Hour UTC', fontsize = 15)
plt.ylabel('BTC PRICE', fontsize = 15)
plt.plot(x,y2, linewidth=5,color='black',label='BTC')
plt.plot(x,y3,'--', linewidth=8,color='darkorange',label='PREDICTED')
plt.legend(frameon=False, loc='upper right', ncol=2)
plt.grid()
plt.rcParams['legend.fontsize'] = 20
plt.tight_layout()
plt.show()